# Circuit Library — 100 built-in quantum circuits
Explore the full catalogue and simulate any circuit with Qiskit Aer.


In [ ]:
!pip install -q knitweb-lens qiskit qiskit-aer matplotlib

In [ ]:
from knitweb_lens import library
from knitweb_lens.library import domains
import pandas as pd

lib = library()
rows = [{'name': c.meta.name, 'domain': c.meta.domain,
         'qubits': c.meta.qubits, 'tags': ', '.join(c.meta.tags),
         'description': c.meta.description}
        for c in lib.values()]
df = pd.DataFrame(rows)
print(f'Total: {len(df)} circuits across {len(domains())} domains')
df.groupby('domain')['name'].count()

In [ ]:
# Display circuits table per domain
for d in domains():
    sub = df[df['domain'] == d][['name','qubits','description']]
    print(f'\n──── {d.upper()} ({len(sub)}) ────')
    for _, row in sub.iterrows():
        print(f'  {row["name"]:<28} q={row["qubits"]}  {row["description"][:50]}')

In [ ]:
# Simulate Bell state with Qiskit AER
from qiskit import QuantumCircuit as QiskitCircuit, transpile
from qiskit_aer import AerSimulator
from knitweb_lens import library

lib = library()
bell = lib['bell_phi_plus']

# Parse QASM into Qiskit circuit
from qiskit import qasm2
qc = qasm2.loads(bell.qasm)
print(qc.draw())

sim = AerSimulator()
tqc = transpile(qc, sim)
result = sim.run(tqc, shots=1024).result()
counts = result.get_counts()
print(f'\nMeasurement counts: {counts}')
# Should be ~50/50 between |00⟩ and |11⟩

In [ ]:
# Simulate Grover 2-qubit search
grover = lib['grover_2q']
qc_g = qasm2.loads(grover.qasm)
tqc_g = transpile(qc_g, sim)
result_g = sim.run(tqc_g, shots=1024).result()
counts_g = result_g.get_counts()
print(f'Grover 2q counts: {counts_g}')
# Marked state should appear with highest probability

In [ ]:
# Plot histogram of all circuit qubit counts
import matplotlib.pyplot as plt

qubit_counts = [c.meta.qubits for c in lib.values()]
plt.figure(figsize=(10, 4))
plt.hist(qubit_counts, bins=range(1, max(qubit_counts)+2), edgecolor='black', color='#5fd0a5')
plt.xlabel('Number of qubits')
plt.ylabel('Circuit count')
plt.title('knitweb-lens library: circuit size distribution (100 circuits)')
plt.tight_layout()
plt.show()